In [10]:
import os
import json
import pandas as pd

In [11]:
# Set the base directory where all experiment folders are stored
BASE_DIR = "/Users/etlevn/Documents/PycharmProjects/LLM_Trd/exp/3.2"

# The error string to detect in JSON files
ERROR_STRING = "Error: Connection error."

# Max number of rounds
MAX_ROUNDS_MAIN = 20
MAX_ROUNDS_PRAC = 3

# Get all experiment folders (named as timestamps, e.g., 20250326_123456)
experiment_folders = [
    f for f in os.listdir(BASE_DIR)
    if os.path.isdir(os.path.join(BASE_DIR, f)) and f[0].isdigit()
]

In [12]:
# Store the result for each experiment
results = []

for exp in sorted(experiment_folders):
    exp_path = os.path.join(BASE_DIR, exp)
    output_path = os.path.join(exp_path, "output")
    
    # Robust check: if output/r4 exists and has .json, it's a main experiment
    r4_path = os.path.join(output_path, "r4")
    is_practice = not (os.path.exists(r4_path) and any(f.endswith(".json") for f in os.listdir(r4_path)))
    
    max_rounds = MAX_ROUNDS_PRAC if is_practice else MAX_ROUNDS_MAIN

    row = {"exp": exp}

    for r in range(1, MAX_ROUNDS_MAIN + 1):  # Always go up to r20
        rname = f"r{r}"

        if r > max_rounds:
            row[rname] = ""  # Leave blank beyond available rounds
            continue

        round_path = os.path.join(output_path, rname)
        error_count = 0
        if os.path.exists(round_path):
            json_files = [f for f in os.listdir(round_path) if f.endswith(".json")]
            for jf in json_files:
                with open(os.path.join(round_path, jf), "r") as f:
                    try:
                        data = json.load(f)
                        if "raw_response" in data and ERROR_STRING in data["raw_response"]:
                            error_count += 1
                    except:
                        pass
            row[rname] = error_count
        else:
            row[rname] = 0 if not is_practice else ""

    # Add total error count (ttl) across all valid rounds
    row["ttl"] = sum(
        v for k, v in row.items()
        if k.startswith("r") and isinstance(v, int)
    )

    results.append(row)

# Convert results to DataFrame
df = pd.DataFrame(results)
df = df[["exp"] + [f"r{i}" for i in range(1, MAX_ROUNDS_MAIN + 1)] + ["ttl"]]

# Convert numeric cells to int, others (blank) remain as ""
for col in df.columns[1:]:
    df[col] = df[col].apply(lambda x: int(x) if isinstance(x, (int, float)) and x != "" else "")

df

,exp,r1,r2,r3,r4,r5,r6,r7,r8,r9,...,r12,r13,r14,r15,r16,r17,r18,r19,r20,ttl
0,20250328_182215,0,0,0,,,,,,,...,,,,,,,,,,0
1,20250328_182833,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,20250328_190609,0,0,0,,,,,,,...,,,,,,,,,,0
3,20250328_191303,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,20250328_194859,0,0,0,,,,,,,...,,,,,,,,,,0
5,20250328_195529,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,20250328_203232,0,0,0,,,,,,,...,,,,,,,,,,0
7,20250328_203902,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,20250328_211852,0,0,0,,,,,,,...,,,,,,,,,,0
9,20250328_212927,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [13]:
df.to_csv("/Users/etlevn/Desktop/3.2.csv")